In [ ]:
# !pip install implicit lightgbm xgboost catboost sentence_transformers networkx
import pandas as pd
import numpy as np
import os
import re
import gc
import networkx as nx
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from catboost import CatBoostRanker, CatBoostClassifier, CatBoostRegressor, Pool
from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.preprocessing import MultiLabelBinarizer, normalize, StandardScaler
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold
from scipy.stats import entropy
from tqdm import tqdm
import warnings
import implicit
from scipy.sparse import csr_matrix
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 500)


PATH = '/kaggle/input/by-pages-ai/participants/participants/'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔌 Neural Network Device: {DEVICE}")

print("Загрузка данных")
try:
    users = pd.read_csv(PATH + 'data/users.csv')
    editions_raw = pd.read_csv(PATH + 'data/editions.csv')
    authors = pd.read_csv(PATH + 'data/authors.csv')
    genres = pd.read_csv(PATH + 'data/genres.csv')
    book_genres = pd.read_csv(PATH + 'data/book_genres.csv')
    interactions = pd.read_csv(PATH + 'data/interactions.csv', parse_dates=['event_ts'])
    candidates = pd.read_csv(PATH + 'submit/candidates.csv')
    targets = pd.read_csv(PATH + 'submit/targets.csv')
except:
    users = pd.read_csv('users.csv')
    editions_raw = pd.read_csv('editions.csv')
    authors = pd.read_csv('authors.csv')
    genres = pd.read_csv('genres.csv')
    book_genres = pd.read_csv('book_genres.csv')
    interactions = pd.read_csv('interactions.csv', parse_dates=['event_ts'])
    candidates = pd.read_csv('candidates.csv')
    targets = pd.read_csv('targets.csv')

print("Parsing Series and Text")
def extract_series_num(title):
    if not isinstance(title, str): return 0
    match = re.search(r'(?:кн\.|книга|том|часть|part|vol|#|no)\s*\.?\s*(\d+)', title, re.IGNORECASE)
    if match: return int(match.group(1))
    return 0
editions_raw['series_num'] = editions_raw['title'].apply(extract_series_num)
editions_raw['full_desc'] = editions_raw['title'].fillna('') + " " + editions_raw['description'].fillna('')
editions_raw['desc_len'] = editions_raw['full_desc'].apply(len)

EMB_PATH = 'bert_embeddings.npy'
if os.path.exists(EMB_PATH):
    print(f" Загрузка эмбеддингов")
    embeddings = np.load(EMB_PATH)
else:
    print("  Генерируем эмбеддинги")
    txt_model = SentenceTransformer('cointegrated/rubert-tiny2')
    embeddings = txt_model.encode(editions_raw['full_desc'].tolist(), show_progress_bar=True, batch_size=128)
    embeddings = normalize(embeddings)

edition_to_vector = dict(zip(editions_raw['edition_id'], embeddings))

print("   ...Clustering")
kmeans = KMeans(n_clusters=60, random_state=42, n_init=10)
editions_raw['cluster_id'] = kmeans.fit_predict(embeddings)
edition_to_cluster = editions_raw.set_index('edition_id')['cluster_id'].to_dict()
edition_to_desc_len = editions_raw.set_index('edition_id')['desc_len'].to_dict()

print("   ...PCA")
pca = PCA(n_components=32, random_state=42)
pca_embeddings = pca.fit_transform(embeddings)
pca_cols = [f'txt_pca_{i}' for i in range(32)]
pca_df = pd.DataFrame(pca_embeddings, columns=pca_cols)
editions_raw = pd.concat([editions_raw, pca_df], axis=1)

edition_to_author = editions_raw.set_index('edition_id')['author_id'].to_dict()
edition_to_publisher = editions_raw.set_index('edition_id')['publisher_id'].to_dict()
edition_to_language = editions_raw.set_index('edition_id')['language_id'].to_dict()
edition_to_book = editions_raw.set_index('edition_id')['book_id'].to_dict()
edition_to_series_num = editions_raw.set_index('edition_id')['series_num'].to_dict()
edition_to_year = editions_raw.set_index('edition_id')['publication_year'].to_dict()
book_to_genres = book_genres.groupby('book_id')['genre_id'].apply(list).to_dict()
edition_to_main_genre = {}
for eid, bid in edition_to_book.items():
    g_list = book_to_genres.get(bid, [])
    if g_list: edition_to_main_genre[eid] = g_list[0]

interactions = interactions.sort_values('event_ts')
cutoff_ts = interactions['event_ts'].max() - pd.Timedelta(days=30)
train_inter = interactions[interactions['event_ts'] < cutoff_ts].copy()
val_inter = interactions[interactions['event_ts'] >= cutoff_ts].copy()


print(" Pre-calculating Stats")
train_extended = train_inter.merge(users, on='user_id', how='left')
gender_pop_map = train_extended.groupby(['gender', 'edition_id']).size().to_dict()
train_extended['age_bin'] = (train_extended['age'] // 10).fillna(-1).astype(int)
age_pop_map = train_extended.groupby(['age_bin', 'edition_id']).size().to_dict()
book_avg_age_train = train_extended.groupby('edition_id')['age'].mean().to_dict()
full_extended = interactions.merge(users, on='user_id', how='left')
book_avg_age_full = full_extended.groupby('edition_id')['age'].mean().to_dict()

user_history_ids = train_inter[train_inter['event_type'].isin([1, 2])].sort_values('event_ts').groupby('user_id')['edition_id'].apply(list).to_dict()
user_weighted_vecs = {}
DECAY_FACTOR = 0.9
for u, hist in tqdm(user_history_ids.items(), desc="Time Decay Vectors"):
    if not hist:
        user_weighted_vecs[u] = np.zeros(embeddings.shape[1])
        continue
    recent = hist[-20:]
    weights = np.array([DECAY_FACTOR ** i for i in range(len(recent))])[::-1]
    vecs = np.array([edition_to_vector[eid] for eid in recent if eid in edition_to_vector])
    if len(vecs) == 0:
        user_weighted_vecs[u] = np.zeros(embeddings.shape[1])
    else:
        if len(vecs) != len(weights): weights = weights[-len(vecs):]
        weighted_sum = np.sum(vecs * weights[:, np.newaxis], axis=0)
        user_weighted_vecs[u] = weighted_sum / (np.sum(weights) + 1e-9)

train_clusters = train_inter.merge(editions_raw[['edition_id', 'cluster_id']], on='edition_id', how='left')
user_cluster_counts = train_clusters.groupby(['user_id', 'cluster_id']).size().to_dict()



def build_swing_matrix(df, alpha=1.0):
    print("   Calculating SWING I2I")
    df = df[df['event_type'].isin([1, 2])].copy()
    user_items = df.groupby('user_id')['edition_id'].apply(list).to_dict()
    item_users = df.groupby('edition_id')['user_id'].apply(list).to_dict()
    swing_score = {}
    relevant_items = {k: v for k, v in item_users.items() if len(v) >= 2}
    for i, (item_i, users_i) in enumerate(tqdm(relevant_items.items(), desc="Swing Loop")):
        users_i_set = set(users_i)
        for user_u in users_i:
            items_u = user_items.get(user_u, [])
            for item_j in items_u:
                if item_i == item_j: continue
                if item_j not in relevant_items: continue
                users_j = relevant_items[item_j]
                common_users = list(users_i_set.intersection(users_j))
                if not common_users: continue
                w = 0.0
                for user_v in common_users:
                    len_v = len(user_items[user_v])
                    w += 1.0 / (alpha + len_v)
                if item_i not in swing_score: swing_score[item_i] = {}
                swing_score[item_i][item_j] = swing_score[item_i].get(item_j, 0) + w
    return swing_score

def add_swing_i2i_feature(df, interactions_df, co_matrix):
    last_books_map = interactions_df[interactions_df['event_type'] == 2].groupby('user_id')['edition_id'].apply(lambda x: list(x)[-5:]).to_dict()
    scores = []
    u_vals = df['user_id'].values
    e_vals = df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        last_items = last_books_map.get(u, [])
        score = 0
        if last_items and e in co_matrix:
            neighbors = co_matrix[e]
            for hist_item in last_items:
                score += neighbors.get(hist_item, 0)
        scores.append(score)
    df['swing_i2i_score'] = scores
    return df


def get_binge_stats(interactions_df):
    df = interactions_df.copy()
    df['author_id'] = df['edition_id'].map(edition_to_author)
    last_author_ts = df.groupby(['user_id', 'author_id'])['event_ts'].max().to_dict()
    return last_author_ts

def add_binge_features(target_df, last_author_ts, edition_to_author, current_ts):
    days_diff_list = []
    u_vals = target_df['user_id'].values
    e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        auth = edition_to_author.get(e)
        if not auth:
            days_diff_list.append(-1)
            continue
        last_ts = last_author_ts.get((u, auth))
        if last_ts is None:
            days_diff_list.append(999) 
        else:
            diff = (current_ts - last_ts).days
            days_diff_list.append(max(0, diff))
    target_df['days_since_last_author_read'] = days_diff_list
    target_df['is_binge_candidate'] = (target_df['days_since_last_author_read'] <= 3).astype(int)
    return target_df

def get_abandonment_stats(interactions_df, editions_df):
    df = interactions_df.copy()
    df['author_id'] = df['edition_id'].map(edition_to_author)
    df['series_num'] = df['edition_id'].map(edition_to_series_num)
    user_author_max_series = df.groupby(['user_id', 'author_id'])['series_num'].max().to_dict()
    return user_author_max_series

def add_abandonment_feature(target_df, user_author_max_series, edition_to_author):
    is_abandoned = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        auth = edition_to_author.get(e)
        if not auth:
            is_abandoned.append(0); continue
        max_s = user_author_max_series.get((u, auth), 0)
        is_abandoned.append(1 if max_s >= 1 else 0)
    target_df['has_started_author'] = is_abandoned
    return target_df

def get_entropy_stats(interactions_df, book_to_genres, edition_to_book):
    u_hist = interactions_df[['user_id', 'edition_id']].copy()
    u_hist['genres'] = u_hist['edition_id'].map(lambda x: book_to_genres.get(edition_to_book.get(x), []))
    u_hist_exploded = u_hist.explode('genres').dropna(subset=['genres'])
    u_counts = u_hist_exploded.groupby('user_id')['genres'].value_counts(normalize=True).unstack(fill_value=0)
    ents = u_counts.apply(entropy, axis=1).to_dict()
    return ents

def add_entropy_feature(target_df, entropy_map):
    target_df['user_genre_entropy'] = target_df['user_id'].map(entropy_map).fillna(1.0)
    return target_df

def get_loyalty_stats(interactions_df, edition_to_author):
    df = interactions_df.copy()
    df['author_id'] = df['edition_id'].map(edition_to_author)
    counts = df.groupby(['user_id', 'author_id']).size().to_dict()
    return counts

def add_loyalty_feature(target_df, loyalty_counts, edition_to_author):
    tiers = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        auth = edition_to_author.get(e)
        cnt = loyalty_counts.get((u, auth), 0)
        if cnt == 0: tiers.append(0)
        elif cnt == 1: tiers.append(1)
        elif cnt == 2: tiers.append(2)
        else: tiers.append(3) 
    target_df['author_loyalty_tier'] = tiers
    return target_df

def get_novelty_stats(interactions_df, edition_to_year):
    df = interactions_df.copy()
    df['year'] = df['edition_id'].map(edition_to_year)
    user_mean_year = df.groupby('user_id')['year'].mean().to_dict()
    return user_mean_year

def add_novelty_feature(target_df, user_mean_year_map, edition_to_year):
    scores = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        u_y = user_mean_year_map.get(u, 2020)
        b_y = edition_to_year.get(e, 2020)
        if pd.isna(b_y): b_y = 2020
        scores.append(abs(u_y - b_y))
    target_df['novelty_diff'] = scores
    return target_df

def get_conversion_stats(interactions_df):
    stats = interactions_df.groupby('edition_id')['event_type'].value_counts().unstack(fill_value=0)
    if 1 in stats.columns and 2 in stats.columns:
        stats['total'] = stats[1] + stats[2]
        stats['conversion'] = (stats[2] + 1) / (stats['total'] + 5)
        return stats['conversion'].to_dict()
    return {}

def add_conversion_feature(target_df, conversion_map):
    target_df['book_conversion_rate'] = target_df['edition_id'].map(conversion_map).fillna(0.2)
    return target_df

def get_stickiness_stats(interactions_df):
    stats = interactions_df.groupby('edition_id')['event_type'].value_counts().unstack(fill_value=0)
    total = stats.sum(axis=1)
    if 2 in stats.columns:
        stick = stats[2] / (total + 5)
        return stick.to_dict()
    return {}

def add_stickiness_feature(target_df, stick_map):
    target_df['book_stickiness'] = target_df['edition_id'].map(stick_map).fillna(0)
    return target_df

def get_trend_accel_stats(interactions_df, current_ts):
    d7 = current_ts - pd.Timedelta(days=7)
    d30 = current_ts - pd.Timedelta(days=30)
    pop7 = interactions_df[interactions_df['event_ts'] > d7]['edition_id'].value_counts()
    pop30 = interactions_df[interactions_df['event_ts'] > d30]['edition_id'].value_counts()
    accel = (pop7 / (pop30 + 1)).to_dict()
    return accel

def add_trend_accel_feature(target_df, accel_map):
    target_df['trend_acceleration'] = target_df['edition_id'].map(accel_map).fillna(0)
    return target_df

def get_pub_spec_stats(interactions_df, edition_to_pub, book_to_genres, edition_to_book):
    df = interactions_df[['edition_id']].drop_duplicates()
    df['pub'] = df['edition_id'].map(edition_to_pub)
    df['book'] = df['edition_id'].map(edition_to_book)
    b2g = {b: gs[0] for b, gs in book_to_genres.items() if gs}
    df['genre'] = df['book'].map(b2g)
    df = df.dropna()
    pg_counts = df.groupby(['pub', 'genre']).size()
    p_totals = df.groupby('pub').size()
    pg_map = (pg_counts / p_totals).to_dict()
    return pg_map, b2g 

def add_pub_spec_feature(target_df, pg_map, edition_to_pub, edition_to_book, b2g_map):
    scores = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        p = edition_to_pub.get(e)
        b = edition_to_book.get(e)
        g = b2g_map.get(b)
        if p and g: scores.append(pg_map.get((p, g), 0))
        else: scores.append(0)
    target_df['pub_specialist_score'] = scores
    return target_df

def get_graph_scores(interactions_df):
    df = interactions_df.sort_values(['user_id', 'event_ts'])
    df['next_edition'] = df.groupby('user_id')['edition_id'].shift(-1)
    pairs = df.dropna(subset=['next_edition'])[['edition_id', 'next_edition']]
    edges = pairs.groupby(['edition_id', 'next_edition']).size().reset_index(name='weight')
    G = nx.DiGraph()
    G.add_weighted_edges_from(edges.values)
    try: pr = nx.pagerank(G, alpha=0.85)
    except: pr = {}
    try: hubs, authorities = nx.hits(G, max_iter=50, normalized=True)
    except: hubs, authorities = {}, {}
    return pr, hubs, authorities

def add_graph_features(target_df, pr_map, hub_map, auth_map):
    target_df['graph_pagerank'] = target_df['edition_id'].map(pr_map).fillna(0)
    target_df['graph_hub'] = target_df['edition_id'].map(hub_map).fillna(0)
    target_df['graph_authority'] = target_df['edition_id'].map(auth_map).fillna(0)
    return target_df

def get_conformity_stats(interactions_df):
    item_pop = interactions_df['edition_id'].value_counts()
    item_pop_log = np.log1p(item_pop).to_dict()
    temp = interactions_df[['user_id', 'edition_id']].copy()
    temp['pop'] = temp['edition_id'].map(item_pop_log)
    user_mean_pop = temp.groupby('user_id')['pop'].mean().to_dict()
    return item_pop_log, user_mean_pop

def add_conformity_feature(target_df, item_pop_map, user_pop_map):
    scores = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        u_p = user_pop_map.get(u, 5.0); i_p = item_pop_map.get(e, 0.0)
        scores.append(1.0 / (abs(u_p - i_p) + 1.0))
    target_df['conformity_score'] = scores
    return target_df

def get_completion_stats(interactions_df):
    stats = interactions_df.groupby('edition_id')['event_type'].value_counts().unstack(fill_value=0)
    if 2 in stats.columns:
        reads = stats[2]; wishes = stats.get(1, 0)
        rate = reads / (reads + wishes + 5)
        return rate.to_dict()
    return {}

def add_completion_feature(target_df, completion_map):
    target_df['completion_rate'] = target_df['edition_id'].map(completion_map).fillna(0)
    return target_df

def get_demo_stats(interactions_df, users_df):
    df = interactions_df.merge(users_df[['user_id', 'gender', 'age']], on='user_id')
    df['age_bin'] = (df['age'] // 10).fillna(-1).astype(int)
    group_counts = df.groupby(['gender', 'age_bin', 'edition_id']).size().reset_index(name='cnt')
    group_totals = df.groupby(['gender', 'age_bin']).size().to_dict()
    demo_map = {}
    for row in group_counts.itertuples():
        key = (row.gender, row.age_bin)
        if key not in demo_map: demo_map[key] = {}
        demo_map[key][row.edition_id] = row.cnt / group_totals.get(key, 1)
    return demo_map

def add_demo_bestseller_feature(target_df, users_df, demo_map):
    temp = target_df[['user_id']].merge(users_df[['user_id', 'gender', 'age']], on='user_id', how='left')
    temp['age_bin'] = (temp['age'] // 10).fillna(-1).astype(int)
    scores = []
    for g, a, e in zip(temp['gender'], temp['age_bin'], target_df['edition_id']):
        key = (g, a); scores.append(demo_map.get(key, {}).get(e, 0))
    target_df['demo_bestseller_score'] = scores
    return target_df

def get_rating_weight(r):
    if pd.isna(r): return 1.0
    if r >= 4: return 2.0
    if r == 3: return 1.0
    return 0.2

def build_rating_i2i_matrix(interactions_df):
    df = interactions_df[['user_id', 'edition_id', 'rating', 'event_ts']].copy()
    df['w'] = df['rating'].apply(get_rating_weight)
    df = df.sort_values(['user_id', 'event_ts'])
    df = df.groupby('user_id').tail(30)
    merged = df.merge(df, on='user_id')
    merged = merged[merged['edition_id_x'] != merged['edition_id_y']]
    merged['pair_w'] = merged['w_x'] * merged['w_y']
    co_weights = merged.groupby(['edition_id_x', 'edition_id_y'])['pair_w'].sum().to_dict()
    return co_weights

def add_rating_i2i_feature(target_df, history_source_df, co_weights, n_last=10):
    user_hist = history_source_df.sort_values('event_ts').groupby('user_id')['edition_id'].apply(lambda x: list(x)[-n_last:]).to_dict()
    scores = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        hist = user_hist.get(u, [])
        score = 0
        for h in hist: score += co_weights.get((h, e), 0)
        scores.append(score)
    target_df['rating_i2i_score'] = scores
    return target_df

def get_cohort_map(interactions_df, editions_df, n_clusters=20):
    users = interactions_df['user_id'].astype('category')
    items = interactions_df['edition_id'].astype('category')
    row = users.cat.codes; col = items.cat.codes
    data = np.ones(len(interactions_df))
    sparse_matrix = csr_matrix((data, (row, col)))
    svd = TruncatedSVD(n_components=16, random_state=42)
    user_vecs = svd.fit_transform(sparse_matrix)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(user_vecs)
    user_to_cluster = dict(zip(users.cat.categories, labels))
    df = interactions_df.copy()
    df['cluster'] = df['user_id'].map(user_to_cluster)
    cluster_item_counts = df.groupby(['cluster', 'edition_id']).size().reset_index(name='cnt')
    cluster_totals = df.groupby('cluster').size().to_dict()
    cluster_pop_map = {}
    for row in cluster_item_counts.itertuples():
        if row.cluster not in cluster_pop_map: cluster_pop_map[row.cluster] = {}
        cluster_pop_map[row.cluster][row.edition_id] = row.cnt / cluster_totals.get(row.cluster, 1)
    return user_to_cluster, cluster_pop_map

def add_cohort_feature(target_df, user_to_cluster, cluster_pop_map):
    scores = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        c = user_to_cluster.get(u, -1)
        if c == -1: scores.append(0)
        else: scores.append(cluster_pop_map.get(c, {}).get(e, 0))
    target_df['cohort_score'] = scores
    return target_df

def get_velocity_stats(interactions_df, edition_to_pub, book_to_genres, edition_to_book):
    df = interactions_df.copy(); max_date = df['event_ts'].max()
    df['days_diff'] = (max_date - df['event_ts']).dt.days; df['weight'] = 0.99 ** df['days_diff']
    df['publisher_id'] = df['edition_id'].map(edition_to_pub)
    user_pub_vel = df.groupby(['user_id', 'publisher_id'])['weight'].sum().to_dict()
    df['book_id'] = df['edition_id'].map(edition_to_book)
    user_genre_vel = {}
    u_b_w = df[['user_id', 'book_id', 'weight']].values
    for u, b, w in u_b_w:
        gs = book_to_genres.get(b, [])
        if not gs: continue
        if u not in user_genre_vel: user_genre_vel[u] = {}
        for g in gs: user_genre_vel[u][g] = user_genre_vel[u].get(g, 0) + w
    return user_pub_vel, user_genre_vel

def add_velocity_features(target_df, user_pub_vel, user_genre_vel, edition_to_pub, edition_to_book, book_to_genres):
    pub_scores = []; gen_scores = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        p = edition_to_pub.get(e)
        pub_scores.append(user_pub_vel.get((u, p), 0) if p else 0)
        b = edition_to_book.get(e); gs = book_to_genres.get(b, [])
        if not gs: gen_scores.append(0)
        else:
            u_prof = user_genre_vel.get(u, {})
            gen_scores.append(sum([u_prof.get(g, 0) for g in gs]))
    target_df['velocity_publisher'] = pub_scores; target_df['velocity_genre'] = gen_scores
    return target_df

def get_author_cluster_stats(interactions_df, edition_to_cluster, edition_to_author):
    df = interactions_df.copy(); df['cluster'] = df['edition_id'].map(edition_to_cluster)
    df['author'] = df['edition_id'].map(edition_to_author)
    clust_cnt = df['cluster'].value_counts().to_dict()
    auth_clust_cnt = df.groupby(['cluster', 'author']).size().to_dict()
    return clust_cnt, auth_clust_cnt

def add_author_cluster_feature(target_df, c_cnt, ac_cnt, edition_to_cluster, edition_to_author):
    scores = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        c = edition_to_cluster.get(e, -1); a = edition_to_author.get(e, -1)
        if c == -1 or a == -1: scores.append(0.0); continue
        count = ac_cnt.get((c, a), 0); total = c_cnt.get(c, 1)
        scores.append(count / (total + 10)) 
    target_df['author_cluster_score'] = scores
    return target_df

def add_age_distance_feature(target_df, users_df, book_avg_age_map):
    user_ages = users_df.set_index('user_id')['age'].to_dict()
    scores = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        u_age = user_ages.get(u); b_age = book_avg_age_map.get(e)
        if pd.isna(u_age) or pd.isna(b_age): scores.append(0.5)
        else: scores.append(1.0 / (abs(u_age - b_age) + 1.0))
    target_df['age_distance_score'] = scores
    return target_df

def get_rarity_stats(interactions_df):
    total = interactions_df['user_id'].nunique()
    book_cnt = interactions_df['edition_id'].value_counts().to_dict()
    idf = {k: np.log(total / (v + 1)) for k, v in book_cnt.items()}
    temp = interactions_df[['user_id', 'edition_id']].copy()
    temp['idf'] = temp['edition_id'].map(idf)
    user_mean = temp.groupby('user_id')['idf'].mean().to_dict()
    return idf, user_mean

def add_rarity_feature(target_df, book_idf, user_mean_idf):
    scores = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        u_r = user_mean_idf.get(u, 5.0); i_r = book_idf.get(e, 5.0)
        scores.append(1.0 / (abs(u_r - i_r) + 1.0))
    target_df['rarity_match'] = scores
    return target_df

def add_cluster_affinity(target_df, history_clusters_df, edition_to_cluster):
    user_cluster_counts = history_clusters_df.groupby(['user_id', 'cluster_id']).size().to_dict()
    user_total_reads = history_clusters_df.groupby('user_id').size().to_dict()
    scores = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        c_id = edition_to_cluster.get(e, -1)
        if c_id == -1: scores.append(0.0); continue
        cnt = user_cluster_counts.get((u, c_id), 0)
        total = user_total_reads.get(u, 1)
        scores.append(cnt / total)
    target_df['cluster_affinity'] = scores
    return target_df

def get_svd_vectors(df, n_components=32):
    users = df['user_id'].astype('category'); items = df['edition_id'].astype('category')
    row = users.cat.codes; col = items.cat.codes
    sparse_matrix = csr_matrix((np.ones(len(df)), (row, col)))
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    svd.fit(sparse_matrix); item_vecs = svd.components_.T
    return dict(zip(items.cat.categories, item_vecs))

def add_svd_feature(target_df, history_source_df, item_vecs_map):
    user_hist_map = history_source_df.groupby('user_id')['edition_id'].apply(list).to_dict()
    user_svd_vecs = {}
    for u, items in user_hist_map.items():
        vecs = [item_vecs_map[i] for i in items if i in item_vecs_map]
        if vecs: user_svd_vecs[u] = np.mean(vecs, axis=0)
    scores = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        u_vec = user_svd_vecs.get(u); i_vec = item_vecs_map.get(e)
        if u_vec is not None and i_vec is not None: scores.append(np.dot(u_vec, i_vec))
        else: scores.append(0.0)
    target_df['svd_similarity'] = scores
    return target_df

def get_pop_stats(df, cutoff_date):
    pop_global = df['edition_id'].value_counts().to_dict()
    recent_start = cutoff_date - pd.Timedelta(days=30)
    pop_recent = df[df['event_ts'] >= recent_start]['edition_id'].value_counts().to_dict()
    return pop_global, pop_recent

def add_trend_features(target_df, pop_global, pop_recent):
    target_df['pop_global_log'] = target_df['edition_id'].map(pop_global).fillna(0).apply(np.log1p)
    target_df['pop_recent_log'] = target_df['edition_id'].map(pop_recent).fillna(0).apply(np.log1p)
    target_df['pop_trend'] = (target_df['edition_id'].map(pop_recent).fillna(0) + 1) / (target_df['edition_id'].map(pop_global).fillna(0) + 10)
    return target_df

def add_author_recency(target_df, interactions_df, edition_to_author, current_ts):
    temp = interactions_df.copy(); temp['author_id'] = temp['edition_id'].map(edition_to_author)
    last_auth_ts = temp.groupby(['user_id', 'author_id'])['event_ts'].max().to_dict()
    recency_scores = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        auth = edition_to_author.get(e, -1)
        if auth == -1: recency_scores.append(0.0); continue
        last_ts = last_auth_ts.get((u, auth))
        if last_ts is None: recency_scores.append(0.0)
        else:
            diff = (current_ts - last_ts).days
            recency_scores.append(1.0 / (max(0, diff) + 1.0))
    target_df['author_recency_score'] = recency_scores
    return target_df

def add_complexity_feature(target_df, user_len_map, edition_to_len):
    scores = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        u_l = user_len_map.get(u, 300); i_l = edition_to_len.get(e, 300)
        scores.append(1.0 / (abs(u_l - i_l) + 1.0))
    target_df['complexity_match'] = scores
    return target_df

def get_complexity_stats(interactions_df, edition_to_len):
    temp = interactions_df[['user_id', 'edition_id']].copy()
    temp['len'] = temp['edition_id'].map(edition_to_len)
    user_mean_len = temp.groupby('user_id')['len'].mean().to_dict()
    return user_mean_len

def get_rating_std_stats(interactions_df):
    return interactions_df.groupby('edition_id')['rating'].std().to_dict()

def add_rating_std_feature(target_df, std_map):
    target_df['rating_std'] = target_df['edition_id'].map(std_map).fillna(0)
    return target_df

def get_user_history_stats(history_df, edition_to_author, edition_to_series):
    df = history_df.copy(); df['author_id'] = df['edition_id'].map(edition_to_author); df['series_num'] = df['edition_id'].map(edition_to_series)
    series_df = df[df['series_num'] > 0]
    user_author_series_max = series_df.groupby(['user_id', 'author_id'])['series_num'].max().to_dict()
    user_read_authors = df.groupby('user_id')['author_id'].apply(set).to_dict()
    return user_author_series_max, user_read_authors

def add_series_features(target_df, user_author_series_max, user_read_authors, edition_to_author, edition_to_series):
    is_next = []; is_sequel_author = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        auth = edition_to_author.get(e, -1); s_num = edition_to_series.get(e, 0)
        last_vol = user_author_series_max.get((u, auth), 0)
        if s_num > 0 and s_num == last_vol + 1: is_next.append(1)
        else: is_next.append(0)
        authors_hist = user_read_authors.get(u, set())
        if auth in authors_hist and s_num > 1: is_sequel_author.append(1)
        else: is_sequel_author.append(0)
    target_df['is_next_in_series'] = is_next
    target_df['is_sequel_of_read_author'] = is_sequel_author
    return target_df

def add_author_hook_feature(df, history_df, edition_to_author):
    last_auth = history_df.sort_values(['user_id', 'event_ts']).groupby('user_id')['edition_id'].last().map(edition_to_author)
    df['is_last_author'] = (df['edition_id'].map(edition_to_author) == df['user_id'].map(last_auth)).astype(int)
    return df

def get_favorite_publishers(history_df, edition_to_publisher):
    df = history_df.copy(); df['publisher_id'] = df['edition_id'].map(edition_to_publisher)
    pub_counts = df.groupby(['user_id', 'publisher_id']).size().reset_index(name='cnt')
    fav_pub = pub_counts.sort_values(['user_id', 'cnt'], ascending=[True, False]).groupby('user_id').first()['publisher_id'].to_dict()
    return fav_pub

def add_publisher_feature(target_df, fav_pub_map, edition_to_publisher):
    is_fav = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        cand_pub = edition_to_publisher.get(e, -1); user_fav = fav_pub_map.get(u, -2)
        if cand_pub != -1 and cand_pub == user_fav: is_fav.append(1)
        else: is_fav.append(0)
    target_df['is_fav_publisher'] = is_fav
    return target_df

def build_user_features(interactions_df, users_df, editions_raw, book_to_genres):
    user_genre_history = (interactions_df.merge(editions_raw[['edition_id', 'book_id']], on='edition_id').assign(genre_id=lambda df: df['book_id'].map(book_to_genres)).explode('genre_id').groupby('user_id')['genre_id'].apply(lambda x: [g for g in x if pd.notna(g)]))
    user_feats = interactions_df.groupby('user_id').agg(user_total_events=('event_type', 'count'), user_wishlist_ratio=('event_type', lambda x: (x == 1).mean()), user_read_ratio=('event_type', lambda x: (x == 2).mean()), user_last_event_ts=('event_ts', 'max'), user_first_event_ts=('event_ts', 'min'), user_avg_rating=('rating', 'mean')).reset_index()
    user_feats = user_feats.merge(users_df[['user_id', 'age', 'gender']], on='user_id', how='left')
    user_feats['user_age'] = user_feats['age'].fillna(user_feats['age'].median())
    user_feats['user_gender'] = user_feats['gender'].fillna(0)
    user_feats['user_genre_history'] = user_feats['user_id'].map(user_genre_history).fillna('').apply(lambda x: x if isinstance(x, list) else [])
    user_feats.fillna(0, inplace=True)
    return user_feats

def build_edition_features(interactions_df, editions_raw, authors_df):
    edition_pop = interactions_df.groupby('edition_id').agg(edition_total_interactions=('user_id', 'count'), edition_unique_users=('user_id', 'nunique'), edition_avg_rating=('rating', 'mean')).reset_index()
    editions_raw = editions_raw.copy(); editions_raw['edition_years_since_pub'] = 2026 - editions_raw['publication_year']
    author_pop = (interactions_df.merge(editions_raw[['edition_id', 'author_id']], on='edition_id').groupby('author_id').size().rename('author_total_interactions'))
    edition_feats = editions_raw.merge(edition_pop, on='edition_id', how='left')
    edition_feats = edition_feats.merge(author_pop, on='author_id', how='left')
    edition_feats.fillna(0, inplace=True)
    base_cols = ['edition_id', 'edition_total_interactions', 'edition_unique_users', 'edition_avg_rating', 'edition_years_since_pub', 'author_total_interactions', 'age_restriction', 'language_id', 'series_num', 'desc_len']
    pca_cols = [c for c in editions_raw.columns if 'txt_pca_' in c]
    return edition_feats[base_cols + pca_cols]

def build_pair_features(df, user_feats, editions_raw, book_to_genres, edition_to_author, edition_to_language):
    df = df.copy()
    if 'user_genre_history' not in df.columns:
        temp = user_feats[['user_id', 'user_genre_history']]
        df = df.merge(temp, on='user_id', how='left')
    safe_history = df['user_genre_history'].apply(lambda x: x if isinstance(x, list) else [])
    all_genres = set(g for genres_list in book_to_genres.values() for g in genres_list)
    mlb = MultiLabelBinarizer(classes=sorted(all_genres))
    user_genre_vecs = pd.DataFrame(mlb.fit_transform(safe_history), columns=mlb.classes_, index=df.index)
    def get_edition_genres(edition_id):
        book_id = edition_to_book.get(edition_id)
        return book_to_genres.get(book_id, []) if book_id else []
    df['edition_genres'] = df['edition_id'].map(get_edition_genres)
    edition_genre_vecs = pd.DataFrame(mlb.transform(df['edition_genres']), columns=mlb.classes_, index=df.index)
    u_mat = user_genre_vecs.values; e_mat = edition_genre_vecs.values
    dot = np.sum(u_mat * e_mat, axis=1); n_u = np.linalg.norm(u_mat, axis=1); n_e = np.linalg.norm(e_mat, axis=1)
    sims = np.zeros(len(df)); mask = (n_u > 0) & (n_e > 0)
    sims[mask] = dot[mask] / (n_u[mask] * n_e[mask])
    df['pair_genre_similarity'] = sims
    df['pair_language_match'] = 0; df['pair_author_match'] = 0
    return df, user_genre_vecs, edition_genre_vecs, mlb, all_genres

def build_session_matrix(interactions_df, time_window_days=3):
    df = interactions_df.sort_values(['user_id', 'event_ts']).copy()
    df['prev_edition'] = df.groupby('user_id')['edition_id'].shift(1)
    df['prev_ts'] = df.groupby('user_id')['event_ts'].shift(1)
    df['days_diff'] = (df['event_ts'] - df['prev_ts']).dt.total_seconds() / (24*3600)
    session_pairs = df[df['days_diff'] <= time_window_days].dropna(subset=['prev_edition'])
    pair_counts = session_pairs.groupby(['prev_edition', 'edition_id']).size().reset_index(name='count')
    pair_counts['weight'] = np.log1p(pair_counts['count'])
    matrix = {}
    for prev, curr, w in pair_counts[['prev_edition', 'edition_id', 'weight']].values:
        prev, curr = int(prev), int(curr)
        matrix[(prev, curr)] = matrix.get((prev, curr), 0) + w
        matrix[(curr, prev)] = matrix.get((curr, prev), 0) + (w * 0.5)
    return matrix

def add_session_feature(target_df, history_source_df, matrix, n_last_items=5):
    last_items_map = (history_source_df.sort_values(['user_id', 'event_ts'])
                      .groupby('user_id')['edition_id'].apply(lambda x: list(x)[-n_last_items:]).to_dict())
    scores = []
    u_vals = target_df['user_id'].values; e_vals = target_df['edition_id'].values
    for u, e in zip(u_vals, e_vals):
        hist = last_items_map.get(u, [])
        if not hist: scores.append(0.0); continue
        s = 0.0
        for h_item in hist: s += matrix.get((h_item, e), 0.0)
        scores.append(s)
    target_df['session_co_score'] = scores
    return target_df

def add_als_score(df, user_factors, item_factors, user_map_inv, item_map_inv):
    scores = []
    if hasattr(user_factors, 'to_numpy'): u_vecs = user_factors.to_numpy()
    else: u_vecs = np.array(user_factors)
    
    if hasattr(item_factors, 'to_numpy'): i_vecs = item_factors.to_numpy()
    else: i_vecs = np.array(item_factors)
    
    for u, e in zip(df['user_id'], df['edition_id']):
        if u in user_map_inv and e in item_map_inv:
            u_idx = user_map_inv[u]
            i_idx = item_map_inv[e]
            scores.append(np.dot(u_vecs[u_idx], i_vecs[i_idx]))
        else: scores.append(0.0)
    df['als_score'] = scores
    return df

def add_demo_popularity(df, users_df):
    temp = df[['user_id']].merge(users_df[['user_id', 'gender', 'age']], on='user_id', how='left')
    temp['age_bin'] = (temp['age'] // 10).fillna(-1).astype(int)
    pop_gender = []; pop_age = []
    for e, g, a in zip(df['edition_id'], temp['gender'], temp['age_bin']):
        pop_gender.append(gender_pop_map.get((g, e), 0))
        pop_age.append(age_pop_map.get((a, e), 0))
    df['pop_gender_group'] = pop_gender; df['pop_age_group'] = pop_age
    return df

def add_text_features_advanced(df):
    sim_max = []; sim_mean = []; cluster_cnt = []
    u_ids, e_ids = df['user_id'].values, df['edition_id'].values
    for u, e in zip(u_ids, e_ids):
        c_id = edition_to_cluster.get(e, -1)
        cluster_cnt.append(user_cluster_counts.get((u, c_id), 0) if c_id != -1 else 0)
        cand_vec = edition_to_vector.get(e)
        if cand_vec is None: sim_max.append(0.0); sim_mean.append(0.0); continue
        hist = user_history_ids.get(u, [])[-50:]
        if not hist: sim_max.append(0.0); sim_mean.append(0.0); continue
        hist_vecs = [edition_to_vector[h] for h in hist if h in edition_to_vector]
        if not hist_vecs: sim_max.append(0.0); sim_mean.append(0.0)
        else:
            scores = np.dot(hist_vecs, cand_vec)
            sim_max.append(np.max(scores)); sim_mean.append(np.mean(scores))
    df['txt_sim_max'] = sim_max; df['txt_sim_mean'] = sim_mean; df['txt_cluster_count'] = cluster_cnt
    return df

def add_weighted_history_sim(df):
    sims = []
    u_ids, e_ids = df['user_id'].values, df['edition_id'].values
    for u, e in zip(u_ids, e_ids):
        u_vec = user_weighted_vecs.get(u); e_vec = edition_to_vector.get(e)
        if u_vec is None or e_vec is None: sims.append(0.0)
        else:
            norm_u = np.linalg.norm(u_vec); norm_e = np.linalg.norm(e_vec)
            if norm_u > 0 and norm_e > 0: sims.append(np.dot(u_vec, e_vec) / (norm_u * norm_e))
            else: sims.append(0.0)
    df['sim_time_weighted'] = sims
    return df

def add_temporal_features(df, interactions_train, cutoff_ts):
    recent_start = cutoff_ts - pd.Timedelta(days=30)
    total_counts = interactions_train['edition_id'].value_counts()
    recent_counts = interactions_train[interactions_train['event_ts'] >= recent_start]['edition_id'].value_counts()
    def get_trend(eid):
        tot = total_counts.get(eid, 0); rec = recent_counts.get(eid, 0)
        return rec / tot if tot > 0 else 0.0
    df['book_trend_score'] = df['edition_id'].apply(get_trend)
    user_last_ts = interactions_train.groupby('user_id')['event_ts'].max()
    df['user_recency_days'] = (cutoff_ts - df['user_id'].map(user_last_ts)).dt.days.fillna(365)
    return df

def calc_te_maps(interactions_train, editions_df):
    temp = interactions_train.merge(editions_df[['edition_id', 'author_id', 'publisher_id']], on='edition_id', how='left')
    temp['weight'] = temp['event_type'].map({2: 3, 1: 1}).fillna(1)
    global_mean = temp['weight'].mean(); smoothing = 10
    auth_stats = temp.groupby('author_id')['weight'].agg(['sum', 'count'])
    auth_stats['score'] = (auth_stats['sum'] + global_mean * smoothing) / (auth_stats['count'] + smoothing)
    author_te_map = auth_stats['score'].to_dict()
    pub_stats = temp.groupby('publisher_id')['weight'].agg(['sum', 'count'])
    pub_stats['score'] = (pub_stats['sum'] + global_mean * smoothing) / (pub_stats['count'] + smoothing)
    publisher_te_map = pub_stats['score'].to_dict()
    return author_te_map, publisher_te_map

def add_te_features(df, editions_df, author_te_map, publisher_te_map):
    df['te_author'] = df['edition_id'].map(edition_to_author).map(author_te_map).fillna(1.5)
    df['te_publisher'] = df['edition_id'].map(edition_to_publisher).map(publisher_te_map).fillna(1.5)
    return df

def add_genre_affinity_sum(df, interactions_train, book_to_genres):
    u_hist = interactions_train[['user_id', 'edition_id']].copy()
    u_hist['genres'] = u_hist['edition_id'].map(lambda x: book_to_genres.get(edition_to_book.get(x), []))
    u_hist_exploded = u_hist.explode('genres').dropna(subset=['genres'])
    u_hist_exploded['genres'] = u_hist_exploded['genres'].astype(int)
    user_genre_counts = u_hist_exploded.groupby(['user_id', 'genres']).size().to_dict()
    scores = []
    for u, e in zip(df['user_id'], df['edition_id']):
        bid = edition_to_book.get(e); g_list = book_to_genres.get(bid, [])
        affinity_sum = sum(user_genre_counts.get((u, g), 0) for g in g_list)
        scores.append(affinity_sum)
    df['aff_genre_sum'] = scores
    return df

def add_author_affinity_feature(df, interactions_df, edition_to_author):
    df_temp = interactions_df.copy(); df_temp['author_id'] = df_temp['edition_id'].map(edition_to_author)
    user_author_counts = df_temp.groupby(['user_id', 'author_id']).size().to_dict()
    scores = []
    for u, e in zip(df['user_id'], df['edition_id']):
        auth = edition_to_author.get(e)
        scores.append(user_author_counts.get((u, auth), 0) if auth else 0)
    df['author_affinity_count'] = scores
    return df

print(" Calculating Global Models (ALS, SVD, SWING, HITS)...")
als_users = train_inter['user_id'].astype('category')
als_items = train_inter['edition_id'].astype('category')
user_map_inv = {u: i for i, u in enumerate(als_users.cat.categories)}
item_map_inv = {i_id: i for i, i_id in enumerate(als_items.cat.categories)}
weights = train_inter['event_type'].map({2: 3, 1: 1}).fillna(1).values
sparse_user_item = csr_matrix((weights, (als_users.cat.codes, als_items.cat.codes)))
als_model = implicit.als.AlternatingLeastSquares(factors=64, regularization=0.05, iterations=20, random_state=42)
als_model.fit(sparse_user_item)
if hasattr(als_model, 'to_cpu'): als_model = als_model.to_cpu()
user_factors = als_model.user_factors
item_factors = als_model.item_factors

# SVD
train_svd_map = get_svd_vectors(train_inter)
test_svd_map = get_svd_vectors(interactions)
# HITS/PR
train_pr, train_hubs, train_auths = get_graph_scores(train_inter)
test_pr, test_hubs, test_auths = get_graph_scores(interactions)
# Conformity
train_item_pop, train_user_pop = get_conformity_stats(train_inter)
test_item_pop, test_user_pop = get_conformity_stats(interactions)
# Completion
train_completion = get_completion_stats(train_inter)
test_completion = get_completion_stats(interactions)
# Demo
train_demo_map = get_demo_stats(train_inter, users)
test_demo_map = get_demo_stats(interactions, users)
# Rating I2I
train_rating_i2i = build_rating_i2i_matrix(train_inter)
test_rating_i2i = build_rating_i2i_matrix(interactions)
# Cohorts
train_u2c, train_c_pop = get_cohort_map(train_inter, editions_raw)
test_u2c, test_c_pop = get_cohort_map(interactions, editions_raw)
# Velocity
train_pub_vel, train_gen_vel = get_velocity_stats(train_inter, edition_to_publisher, book_to_genres, edition_to_book)
test_pub_vel, test_gen_vel = get_velocity_stats(interactions, edition_to_publisher, book_to_genres, edition_to_book)
# Auth Cluster
train_c_cnt, train_ac_cnt = get_author_cluster_stats(train_inter, edition_to_cluster, edition_to_author)
test_c_cnt, test_ac_cnt = get_author_cluster_stats(interactions, edition_to_cluster, edition_to_author)
# Rarity
train_idf, train_user_idf = get_rarity_stats(train_inter)
test_idf, test_user_idf = get_rarity_stats(interactions)
# Trend
train_pop_glob, train_pop_rec = get_pop_stats(train_inter, cutoff_ts)
test_pop_glob, test_pop_rec = get_pop_stats(interactions, interactions['event_ts'].max())
# Complexity
train_user_len = get_complexity_stats(train_inter, edition_to_desc_len)
test_user_len = get_complexity_stats(interactions, edition_to_desc_len)
# Rating Std
train_rating_std = get_rating_std_stats(train_inter)
test_rating_std = get_rating_std_stats(interactions)
# Stickiness
train_stickiness = get_stickiness_stats(train_inter)
test_stickiness = get_stickiness_stats(interactions)
# Trend Accel
train_trend_accel = get_trend_accel_stats(train_inter, cutoff_ts)
test_trend_accel = get_trend_accel_stats(interactions, interactions['event_ts'].max())
# Pub Spec
train_pg, train_pt = get_pub_spec_stats(train_inter, edition_to_publisher, book_to_genres, edition_to_book)
test_pg, test_pt = get_pub_spec_stats(interactions, edition_to_publisher, book_to_genres, edition_to_book)

# STATS
train_binge = get_binge_stats(train_inter)
test_binge = get_binge_stats(interactions)
train_abandon = get_abandonment_stats(train_inter, editions_raw)
test_abandon = get_abandonment_stats(interactions, editions_raw)
train_entropy = get_entropy_stats(train_inter, book_to_genres, edition_to_book)
test_entropy = get_entropy_stats(interactions, book_to_genres, edition_to_book)
train_loyalty = get_loyalty_stats(train_inter, edition_to_author)
test_loyalty = get_loyalty_stats(interactions, edition_to_author)
train_novelty = get_novelty_stats(train_inter, edition_to_year)
test_novelty = get_novelty_stats(interactions, edition_to_year)
train_conversion = get_conversion_stats(train_inter)
test_conversion = get_conversion_stats(interactions)

# Base Maps
swing_matrix_train = build_swing_matrix(train_inter)
swing_matrix_test = build_swing_matrix(interactions)
author_te_map, publisher_te_map = calc_te_maps(train_inter, editions_raw)
session_matrix = build_session_matrix(interactions, time_window_days=3)
train_inter_clustered = train_inter.merge(editions_raw[['edition_id', 'cluster_id']], on='edition_id', how='left')
full_inter_clustered = interactions.merge(editions_raw[['edition_id', 'cluster_id']], on='edition_id', how='left')

# ASSEMBLER FUNCTION
def apply_all_base_features(df, swing_matrix_source, is_train=True):
    df = add_swing_i2i_feature(df, train_inter if is_train else interactions, swing_matrix_source)
    df = add_als_score(df, user_factors, item_factors, user_map_inv, item_map_inv)
    df = add_author_affinity_feature(df, train_inter, edition_to_author)
    df = add_temporal_features(df, train_inter, cutoff_ts)
    df = add_te_features(df, editions_raw, author_te_map, publisher_te_map)
    df = add_genre_affinity_sum(df, train_inter, book_to_genres)
    df = add_demo_popularity(df, users)
    df = add_text_features_advanced(df)
    df = add_weighted_history_sim(df)
    def f(t):
        if pd.isna(t): return 0.0
        if hasattr(t, 'timestamp'): return t.timestamp()
        try: return float(t)
        except: return 0.0
    df['user_last_event_ts'] = df['user_last_event_ts'].apply(f)
    df['user_first_event_ts'] = df['user_first_event_ts'].apply(f)
    return df

print(" Сборка Train")
user_feats = build_user_features(train_inter, users, editions_raw, book_to_genres)
edition_feats = build_edition_features(train_inter, editions_raw, authors)
val_pos = val_inter[['user_id', 'edition_id']].drop_duplicates()
val_pos['target'] = 1
val_users = val_pos['user_id'].unique()
val_candidates = candidates[candidates['user_id'].isin(val_users)].copy()
val_neg = val_candidates.merge(val_pos, on=['user_id', 'edition_id'], how='left', indicator=True).query('_merge == "left_only"')[['user_id', 'edition_id']].drop_duplicates()
val_neg['target'] = 0
val_neg_sampled = val_neg.sample(n=min(len(val_pos) * 5, len(val_neg)), random_state=42)
train_df = pd.concat([val_pos, val_neg_sampled], ignore_index=True)

train_df = train_df.merge(user_feats.drop(columns=['user_genre_history']), on='user_id', how='left')
train_df = train_df.merge(edition_feats, on='edition_id', how='left')
train_df, user_genre_vecs, edition_genre_vecs, mlb, all_genres = build_pair_features(train_df, user_feats, editions_raw, book_to_genres, edition_to_author, edition_to_language)

print("Calculating Base Features for Train")
train_df = apply_all_base_features(train_df, swing_matrix_train, is_train=True)
print(" Calculating SESSION I2I Feature for Train")
train_df = add_session_feature(train_df, train_inter, session_matrix)

print(" Calculating ASTRUM Features for Train")
train_df = add_graph_features(train_df, train_pr, train_hubs, train_auths)
train_df = add_conformity_feature(train_df, train_item_pop, train_user_pop)
train_df = add_completion_feature(train_df, train_completion)
train_df = add_demo_bestseller_feature(train_df, users, train_demo_map)
train_df = add_rating_i2i_feature(train_df, train_inter, train_rating_i2i)
train_df = add_cohort_feature(train_df, train_u2c, train_c_pop)
train_df = add_velocity_features(train_df, train_pub_vel, train_gen_vel, edition_to_publisher, edition_to_book, book_to_genres)
train_df = add_author_cluster_feature(train_df, train_c_cnt, train_ac_cnt, edition_to_cluster, edition_to_author)
train_df = add_age_distance_feature(train_df, users, book_avg_age_train)
train_df = add_rarity_feature(train_df, train_idf, train_user_idf)
train_df = add_cluster_affinity(train_df, train_inter_clustered, edition_to_cluster)
train_df = add_svd_feature(train_df, train_inter, train_svd_map)
train_df = add_trend_features(train_df, train_pop_glob, train_pop_rec)
train_df = add_author_recency(train_df, train_inter, edition_to_author, current_ts=cutoff_ts)
train_series_max, train_read_authors = get_user_history_stats(train_inter, edition_to_author, edition_to_series_num)
train_df = add_series_features(train_df, train_series_max, train_read_authors, edition_to_author, edition_to_series_num)
train_df = add_author_hook_feature(train_df, train_inter, edition_to_author)
train_fav_pubs = get_favorite_publishers(train_inter, edition_to_publisher)
train_df = add_publisher_feature(train_df, train_fav_pubs, edition_to_publisher)
train_df = add_rating_std_feature(train_df, train_rating_std)
train_df = add_complexity_feature(train_df, train_user_len, edition_to_desc_len)

train_df = add_stickiness_feature(train_df, train_stickiness)
train_df = add_trend_accel_feature(train_df, train_trend_accel)
train_df = add_pub_spec_feature(train_df, train_pg, edition_to_publisher, edition_to_book, train_pt)
train_df = add_binge_features(train_df, train_binge, edition_to_author, cutoff_ts)
train_df = add_abandonment_feature(train_df, train_abandon, edition_to_author)
train_df = add_entropy_feature(train_df, train_entropy)
train_df = add_loyalty_feature(train_df, train_loyalty, edition_to_author)
train_df = add_novelty_feature(train_df, train_novelty, edition_to_year)
train_df = add_conversion_feature(train_df, train_conversion)

feature_cols_base = [col for col in train_df.columns if col not in [
    'user_id', 'edition_id', 'target', 'user_genre_history', 'edition_genres', 'genre_id', 'genre_name'
]]
for x in feature_cols_base:
    if train_df[x].dtype != 'object':
        train_df[x] = train_df[x].fillna(train_df[x].mean())

train_df['group_id'] = train_df['user_id'].factorize()[0]
train_df = train_df.sort_values('user_id')

train_inter_unique = train_inter.sort_values('event_type', ascending=False).drop_duplicates(subset=['user_id', 'edition_id'])
train_df_stack = train_df.merge(train_inter_unique[['user_id', 'edition_id', 'event_type']], on=['user_id', 'edition_id'], how='left')
train_df_stack['reg_target'] = train_df_stack['event_type'].map({2: 3, 1: 1}).fillna(0)


print("\n STARTING FULL HYBRID STACKING")

class TabularNN(nn.Module):
    def __init__(self, input_dim):
        super(TabularNN, self).__init__()
        self.layer1 = nn.Sequential(nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3))
        self.layer2 = nn.Sequential(nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2))
        self.layer3 = nn.Sequential(nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.1))
        self.output = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x): return self.sigmoid(self.output(self.layer3(self.layer2(self.layer1(x)))))

def train_predict_nn(X_train, y_train, X_val, epochs=7):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    train_ds = TensorDataset(torch.tensor(X_train_scaled, dtype=torch.float32), 
                             torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1))
    train_dl = DataLoader(train_ds, batch_size=2048, shuffle=True)
    model = TabularNN(X_train.shape[1]).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=0.001); criterion = nn.BCELoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad(); loss = criterion(model(xb), yb); loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad(): preds = model(torch.tensor(X_val_scaled, dtype=torch.float32).to(DEVICE)).cpu().numpy().flatten()
    return preds, model, scaler

train_df['meta_cb_prob'] = 0.0; train_df['meta_cb_reg'] = 0.0
train_df['meta_lgbm_prob'] = 0.0; train_df['meta_lgbm_reg'] = 0.0
train_df['meta_xgb_prob'] = 0.0; train_df['meta_xgb_reg'] = 0.0
train_df['meta_nn_prob'] = 0.0; train_df['meta_div_dist_to_top'] = 0.0
train_df['meta_uncertainty'] = 0.0

gkf = GroupKFold(n_splits=3)
users_for_split = train_df['user_id'].values

l1_cb_params = {'iterations': 500, 'depth': 8, 'learning_rate': 0.05, 'verbose': 0, 'random_state': 42}
l1_lgbm_params = {'n_estimators': 500, 'max_depth':  8, 'learning_rate': 0.05, 'verbose': -1, 'random_state': 42}
l1_xgb_params = {'n_estimators': 500, 'max_depth':  8, 'learning_rate': 0.05, 'tree_method': 'hist', 'random_state': 42}

for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=users_for_split)):
    print(f"   ...Processing Fold {fold+1}/3...")
    X_train = train_df.iloc[train_idx][feature_cols_base]
    y_train_bin = train_df.iloc[train_idx]['target']
    y_train_reg = train_df_stack.iloc[train_idx]['reg_target']
    X_val = train_df.iloc[val_idx][feature_cols_base]
    
    cb_clf = CatBoostClassifier(loss_function='Logloss', **l1_cb_params); cb_clf.fit(X_train, y_train_bin)
    train_df.iloc[val_idx, train_df.columns.get_loc('meta_cb_prob')] = cb_clf.predict_proba(X_val)[:, 1]
    cb_reg = CatBoostRegressor(loss_function='RMSE', **l1_cb_params); cb_reg.fit(X_train, y_train_reg)
    train_df.iloc[val_idx, train_df.columns.get_loc('meta_cb_reg')] = cb_reg.predict(X_val)
    lgbm_clf = LGBMClassifier(**l1_lgbm_params); lgbm_clf.fit(X_train, y_train_bin)
    train_df.iloc[val_idx, train_df.columns.get_loc('meta_lgbm_prob')] = lgbm_clf.predict_proba(X_val)[:, 1]
    lgbm_reg = LGBMRegressor(**l1_lgbm_params); lgbm_reg.fit(X_train, y_train_reg)
    train_df.iloc[val_idx, train_df.columns.get_loc('meta_lgbm_reg')] = lgbm_reg.predict(X_val)
    xgb_clf = XGBClassifier(**l1_xgb_params); xgb_clf.fit(X_train, y_train_bin)
    train_df.iloc[val_idx, train_df.columns.get_loc('meta_xgb_prob')] = xgb_clf.predict_proba(X_val)[:, 1]
    xgb_reg = XGBRegressor(**l1_xgb_params); xgb_reg.fit(X_train, y_train_reg)
    train_df.iloc[val_idx, train_df.columns.get_loc('meta_xgb_reg')] = xgb_reg.predict(X_val)
    probs_nn, _, _ = train_predict_nn(X_train, y_train_bin, X_val)
    train_df.iloc[val_idx, train_df.columns.get_loc('meta_nn_prob')] = probs_nn
    all_probs = np.column_stack([train_df.iloc[val_idx]['meta_cb_prob'], train_df.iloc[val_idx]['meta_lgbm_prob'], train_df.iloc[val_idx]['meta_xgb_prob'], probs_nn])
    train_df.iloc[val_idx, train_df.columns.get_loc('meta_uncertainty')] = np.std(all_probs, axis=1)
    fold_res = pd.DataFrame({'user_id': train_df.iloc[val_idx]['user_id'].values, 'prob': train_df.iloc[val_idx]['meta_cb_prob'], 'idx': val_idx})
    user_top_idx_map = fold_res.sort_values('prob', ascending=False).groupby('user_id')['idx'].first().to_dict()
    top_idxs = fold_res['user_id'].map(user_top_idx_map).values
    v_curr, v_top = edition_genre_vecs.iloc[val_idx].values, edition_genre_vecs.iloc[top_idxs].values
    norms_c, norms_t = np.linalg.norm(v_curr, axis=1), np.linalg.norm(v_top, axis=1)
    dots = np.sum(v_curr * v_top, axis=1); sims = np.zeros_like(dots)
    mask = (norms_c > 0) & (norms_t > 0); sims[mask] = dots[mask] / (norms_c[mask] * norms_t[mask])
    train_df.iloc[val_idx, train_df.columns.get_loc('meta_div_dist_to_top')] = 1.0 - sims

print(" Retraining Level 1 Models")
cb_clf_full = CatBoostClassifier(loss_function='Logloss', **l1_cb_params); cb_clf_full.fit(train_df[feature_cols_base], train_df['target'])
cb_reg_full = CatBoostRegressor(loss_function='RMSE', **l1_cb_params); cb_reg_full.fit(train_df[feature_cols_base], train_df_stack['reg_target'])
lgbm_clf_full = LGBMClassifier(**l1_lgbm_params); lgbm_clf_full.fit(train_df[feature_cols_base], train_df['target'])
lgbm_reg_full = LGBMRegressor(**l1_lgbm_params); lgbm_reg_full.fit(train_df[feature_cols_base], train_df_stack['reg_target'])
xgb_clf_full = XGBClassifier(**l1_xgb_params); xgb_clf_full.fit(train_df[feature_cols_base], train_df['target'])
xgb_reg_full = XGBRegressor(**l1_xgb_params); xgb_reg_full.fit(train_df[feature_cols_base], train_df_stack['reg_target'])
_, nn_model_full, nn_scaler_full = train_predict_nn(train_df[feature_cols_base], train_df['target'], train_df[feature_cols_base], epochs=20)


print("\n Training Level 2 (Full Hybrid Ranker + Weights)")
feature_cols_l2 = feature_cols_base + ['meta_cb_prob', 'meta_cb_reg', 'meta_lgbm_prob', 'meta_lgbm_reg', 'meta_xgb_prob', 'meta_xgb_reg', 'meta_nn_prob', 'meta_uncertainty', 'meta_div_dist_to_top']

train_pool = Pool(data=train_df[feature_cols_l2], label=train_df['target'], group_id=train_df['group_id'])
model_l2 = CatBoostRanker(iterations=1300, learning_rate=0.035, depth=6, l2_leaf_reg=5, loss_function='YetiRank', random_seed=42)
model_l2.fit(train_pool, verbose=100)


del train_df, train_df_stack, train_pool, swing_matrix_train
gc.collect()


print("\n Generating Test Candidates")
test_candidates = candidates[candidates['user_id'].isin(targets['user_id'])].copy()

test_candidates = test_candidates.merge(user_feats.drop(columns=['user_genre_history']), on='user_id', how='left')
test_candidates = test_candidates.merge(edition_feats, on='edition_id', how='left')
test_candidates, _, _, _, _ = build_pair_features(test_candidates, user_feats, editions_raw, book_to_genres, edition_to_author, edition_to_language)

print(" Calculating SWING Matrix for Test (Full Interactions)...")
swing_matrix_test = build_swing_matrix(interactions)

print(" Calculating Base Features for Test...")
test_candidates = apply_all_base_features(test_candidates, swing_matrix_test, is_train=False)
test_candidates = add_session_feature(test_candidates, interactions, session_matrix)

print(" Calculating ASTRUM Features for Test...")
full_inter_clustered = interactions.merge(editions_raw[['edition_id', 'cluster_id']], on='edition_id', how='left')
current_test_time = interactions['event_ts'].max()

test_candidates = add_graph_features(test_candidates, test_pr, test_hubs, test_auths) 
test_candidates = add_conformity_feature(test_candidates, test_item_pop, test_user_pop)
test_candidates = add_completion_feature(test_candidates, test_completion)
test_candidates = add_rating_std_feature(test_candidates, test_rating_std)

test_candidates = add_stickiness_feature(test_candidates, test_stickiness)
test_candidates = add_trend_accel_feature(test_candidates, test_trend_accel)
test_candidates = add_pub_spec_feature(test_candidates, test_pg, edition_to_publisher, edition_to_book, test_pt)

test_candidates = add_demo_bestseller_feature(test_candidates, users, test_demo_map)
test_candidates = add_rating_i2i_feature(test_candidates, interactions, test_rating_i2i)
test_candidates = add_cohort_feature(test_candidates, test_u2c, test_c_pop)
test_candidates = add_velocity_features(test_candidates, test_pub_vel, test_gen_vel, edition_to_publisher, edition_to_book, book_to_genres)
test_candidates = add_author_cluster_feature(test_candidates, test_c_cnt, test_ac_cnt, edition_to_cluster, edition_to_author)
test_candidates = add_age_distance_feature(test_candidates, users, book_avg_age_full)
test_candidates = add_rarity_feature(test_candidates, test_idf, test_user_idf)
test_candidates = add_cluster_affinity(test_candidates, full_inter_clustered, edition_to_cluster)
test_candidates = add_svd_feature(test_candidates, interactions, test_svd_map)
test_candidates = add_trend_features(test_candidates, test_pop_glob, test_pop_rec)
test_candidates = add_author_recency(test_candidates, interactions, edition_to_author, current_ts=current_test_time)
test_series_max, test_read_authors = get_user_history_stats(interactions, edition_to_author, edition_to_series_num)
test_candidates = add_series_features(test_candidates, test_series_max, test_read_authors, edition_to_author, edition_to_series_num)
test_candidates = add_author_hook_feature(test_candidates, interactions, edition_to_author)
test_fav_pubs = get_favorite_publishers(interactions, edition_to_publisher)
test_candidates = add_publisher_feature(test_candidates, test_fav_pubs, edition_to_publisher)
test_candidates = add_complexity_feature(test_candidates, test_user_len, edition_to_desc_len)

test_candidates = add_binge_features(test_candidates, test_binge, edition_to_author, current_test_time)
test_candidates = add_abandonment_feature(test_candidates, test_abandon, edition_to_author)
test_candidates = add_entropy_feature(test_candidates, test_entropy)
test_candidates = add_loyalty_feature(test_candidates, test_loyalty, edition_to_author)
test_candidates = add_novelty_feature(test_candidates, test_novelty, edition_to_year)
test_candidates = add_conversion_feature(test_candidates, test_conversion)

test_candidates[feature_cols_base] = test_candidates[feature_cols_base].fillna(0)

print("   ...Applying L1 to Test")
probs_cb = cb_clf_full.predict_proba(test_candidates[feature_cols_base])[:, 1]
test_candidates['meta_cb_prob'] = probs_cb
test_candidates['meta_cb_reg'] = cb_reg_full.predict(test_candidates[feature_cols_base])
probs_lgbm = lgbm_clf_full.predict_proba(test_candidates[feature_cols_base])[:, 1]
test_candidates['meta_lgbm_prob'] = probs_lgbm
test_candidates['meta_lgbm_reg'] = lgbm_reg_full.predict(test_candidates[feature_cols_base])
probs_xgb = xgb_clf_full.predict_proba(test_candidates[feature_cols_base])[:, 1]
test_candidates['meta_xgb_prob'] = probs_xgb
test_candidates['meta_xgb_reg'] = xgb_reg_full.predict(test_candidates[feature_cols_base])

X_test_scaled = nn_scaler_full.transform(test_candidates[feature_cols_base])
with torch.no_grad():
    probs_nn = nn_model_full(torch.tensor(X_test_scaled, dtype=torch.float32).to(DEVICE)).cpu().numpy().flatten()
test_candidates['meta_nn_prob'] = probs_nn

all_probs_test = np.column_stack([probs_cb, probs_lgbm, probs_xgb, probs_nn])
test_candidates['meta_uncertainty'] = np.std(all_probs_test, axis=1)

print("   ...Calc Diversity Meta-Feature (Test)")
test_edition_genres = test_candidates['edition_id'].map(lambda x: book_to_genres.get(edition_to_book.get(x), []))
test_genre_vecs_arr = mlb.transform(test_edition_genres)
temp_test = test_candidates[['user_id', 'meta_cb_prob']].copy()
temp_test['idx'] = range(len(temp_test))
user_top_idx = temp_test.sort_values('meta_cb_prob', ascending=False).groupby('user_id')['idx'].first().to_dict()
top_idxs = temp_test['user_id'].map(user_top_idx).values
vecs_curr = test_genre_vecs_arr
vecs_top = test_genre_vecs_arr[top_idxs]
norms_c, norms_t = np.linalg.norm(vecs_curr, axis=1), np.linalg.norm(vecs_top, axis=1)
dots = np.sum(vecs_curr * vecs_top, axis=1); sims = np.zeros_like(dots)
mask = (norms_c > 0) & (norms_t > 0); sims[mask] = dots[mask] / (norms_c[mask] * norms_t[mask])
test_candidates['meta_div_dist_to_top'] = 1.0 - sims 

print("   ...Predicting L2 Score")
test_candidates['l2_score'] = model_l2.predict(test_candidates[feature_cols_l2])

print("Running SMART MMR v2 (Genre Graph + Sequel Immunity)")

print("   ...Building Genre Correlation Graph")
temp_g = interactions.merge(editions_raw[['edition_id', 'book_id']], on='edition_id').merge(book_genres, on='book_id')
top_g = temp_g['genre_id'].value_counts().head(30).index
temp_g = temp_g[temp_g['genre_id'].isin(top_g)]
genre_matrix = pd.crosstab(temp_g['user_id'], temp_g['genre_id'])
corr_matrix = genre_matrix.corr()

friendly_genres = {}
for g1 in corr_matrix.columns:
    friends = corr_matrix[g1][corr_matrix[g1] > 0.45].index.tolist()
    friendly_genres[g1] = set(friends)

user_genre_diversity = interactions.merge(editions_raw[['edition_id', 'book_id']], on='edition_id') \
                                   .merge(book_genres, on='book_id') \
                                   .groupby('user_id')['genre_id'].nunique()
max_gen = user_genre_diversity.max()
user_diversity_score = (user_genre_diversity / max_gen).to_dict()

def smart_mmr_rerank_v2(user_id, candidates_df, relevance_scores, genre_vectors, base_lambda=0.85, top_k=20):
    u_div = user_diversity_score.get(user_id, 0.2)
    current_lambda = base_lambda
    if u_div < 0.15: current_lambda = 0.96 
    elif u_div > 0.5: current_lambda = 0.80 
        
    if len(candidates_df) <= top_k:
        r = candidates_df.copy(); r['rank'] = range(1, len(r)+1); return r
    
    selected = []
    remaining = list(range(len(candidates_df)))
    
    cand_main_genres = candidates_df['edition_id'].map(edition_to_main_genre).fillna(-1).values
    series_flags = candidates_df['is_next_in_series'].values
    
    first_idx = np.argmax(relevance_scores)
    selected.append(first_idx)
    remaining.remove(first_idx)
    
    while len(selected) < top_k and remaining:
        best_score = -np.inf
        best_idx = None
        
        for idx in remaining:
            rel = relevance_scores[idx]
            max_sim = 0.0
            if selected:
                sims = np.dot(genre_vectors[selected], genre_vectors[idx])
                max_sim = np.max(sims)
                
                if series_flags[idx] == 1:
                    max_sim = 0.0 
                else:
                    g_curr = cand_main_genres[idx]
                    if g_curr != -1:
                        for sel_idx in selected:
                            g_sel = cand_main_genres[sel_idx]
                            if g_sel != -1 and g_curr in friendly_genres.get(g_sel, set()):
                                max_sim *= 0.6 
                                break 
            
            mmr_val = current_lambda * rel - (1 - current_lambda) * max_sim
            if mmr_val > best_score:
                best_score = mmr_val; best_idx = idx
                
        selected.append(best_idx); remaining.remove(best_idx)
        
    ranked = candidates_df.iloc[selected].copy()
    ranked['rank'] = range(1, len(ranked)+1)
    return ranked

submit_rows = []
for user_id, group in tqdm(test_candidates.groupby('user_id'), desc="Smart Reranking"):
    rel = group['l2_score'].values
    if rel.max() != rel.min(): rel = (rel - rel.min()) / (rel.max() - rel.min())
    else: rel = np.zeros_like(rel)
    g_vecs = test_genre_vecs_arr[group.index]
    norms = np.linalg.norm(g_vecs, axis=1, keepdims=True)
    g_vecs_norm = g_vecs / (norms + 1e-9)
    ranked = smart_mmr_rerank_v2(user_id, group.reset_index(drop=True), rel, g_vecs_norm, base_lambda=0.85, top_k=20)
    ranked['user_id'] = user_id
    submit_rows.append(ranked[['user_id', 'edition_id', 'rank']])

submission = pd.concat(submit_rows, ignore_index=True)
submission.to_csv('submission_astrum_agi_ultimate.csv', index=False)
print("Saved submission_astrum_agi_ultimate.csv") # 0,7191889299588444